In [57]:
import itertools
import json
from pathlib import Path
from datetime import datetime
import pandas as pd

pd.set_option("display.max_colwidth", 200)

In [58]:
KEYWORD_CONFIG = {
    "object_categories": {
        "consumer_electronics": [
            "airpods", "earbuds", "headphones", "ipad", "tablet", "macbook"
        ],
        "camera_drone": [
            "camera", "camera lens", "drone", "gopro", "drone battery"
        ],
        "digital_accessories": [
            "charger", "cables", "power bank", "adapter", "usb hub", "ssd", "hard drive"
        ],
        "gaming": [
            "switch", "steam deck", "keyboard", "gaming gear"
        ],
        "valuable_items": [
            "watches", "jewelry", "collectibles", "figurines", "cards", "makeup"
        ],
        "travel_outdoor": [
            "travel gear", "camping gear", "hiking equipment", "carry on essentials", "tools"
        ]
    },

    "scenario_queries": [
        "what's in my bag tech",
        "everyday carry tech EDC",
        "EDC setup gadgets",
        "tech travel setup",
        "travel gadgets packing",
        "camera bag setup",
        "drone travel setup",
        "photography gear setup",
        "minimalist tech setup",
        "desk setup tech gadgets",
        "work from home setup tech",
        "gaming setup accessories",
        "college tech essentials",
        "daily carry gadgets"
    ],

    "problem_queries": [
        "how to protect electronics",
        "how to store cables",
        "how to organize tech accessories",
        "how to carry camera safely",
        "how to store drone",
        "how to pack tech for travel",
        "best way to protect gadgets",
        "how to prevent scratches on devices",
        "safe storage for electronics",
        "how to organize backpack tech"
    ],

    "object_scenario_patterns": [
        "{obj} travel setup",
        "{obj} bag setup",
        "{obj} daily carry",
        "{obj} accessories setup"
    ],

    "object_problem_patterns": [
        "how to protect {obj}",
        "how to store {obj}",
        "best way to carry {obj}",
        "safe storage for {obj}",
        "{obj} scratched easily",
        "{obj} easy to break",
        "{obj} needs a case",
        "no good case for {obj}",
        "{obj} damaged in bag"
    ],

    "product_support_queries": [
        "camera accessories review",
        "drone accessories review",
        "tech organizer review",
        "electronics carrying case review",
        "gadget storage solutions review"
    ],

    "global_trends": [
        "viral tech gadgets",
        "new gadgets you must have",
        "amazon tech finds",
        "cool gadgets you need",
        "latest tech accessories"
    ]
}

In [59]:
config_dir = Path("../../data/shared_data/config")
config_dir.mkdir(parents=True, exist_ok=True)

config_file = config_dir / "keyword_config.json"

with open(config_file, "w", encoding="utf-8") as f:
    json.dump(KEYWORD_CONFIG, f, ensure_ascii=False, indent=2)

print("配置已保存到：", config_file)

配置已保存到： ../../data/shared_data/config/keyword_config.json


In [60]:
def load_keyword_config(config_path: Path) -> dict:
    with open(config_path, "r", encoding="utf-8") as f:
        return json.load(f)

config = load_keyword_config(config_file)

print(config.keys())

dict_keys(['object_categories', 'scenario_queries', 'problem_queries', 'object_scenario_patterns', 'object_problem_patterns', 'product_support_queries', 'global_trends'])


In [61]:
def generate_queries_for_categories(
    config: dict,
    categories: list[str],
    include_global_terms: bool = False
) -> pd.DataFrame:
    rows = []

    scenario_queries = config["scenario_queries"]
    problem_queries = config["problem_queries"]
    object_scenario_patterns = config["object_scenario_patterns"]
    object_problem_patterns = config["object_problem_patterns"]
    product_support_queries = config["product_support_queries"]

    # 1) 通用场景类：优先找高价值评论区
    for q in scenario_queries:
        rows.append({
            "category": "generic",
            "object": None,
            "query": q,
            "query_type": "scenario"
        })

    # 2) 通用问题类：优先找会讨论痛点的评论区
    for q in problem_queries:
        rows.append({
            "category": "generic",
            "object": None,
            "query": q,
            "query_type": "problem"
        })

    # 3) 对象 + 场景
    for category in categories:
        objects = config["object_categories"].get(category, [])

        for obj, pattern in itertools.product(objects, object_scenario_patterns):
            rows.append({
                "category": category,
                "object": obj,
                "query": pattern.format(obj=obj),
                "query_type": "object_scenario"
            })

    # 4) 对象 + 问题
    for category in categories:
        objects = config["object_categories"].get(category, [])

        for obj, pattern in itertools.product(objects, object_problem_patterns):
            rows.append({
                "category": category,
                "object": obj,
                "query": pattern.format(obj=obj),
                "query_type": "object_problem"
            })

    # 5) 产品补充
    for q in product_support_queries:
        rows.append({
            "category": "generic",
            "object": None,
            "query": q,
            "query_type": "product_support"
        })

    # 6) 趋势词
    if include_global_terms:
        for q in config["global_trends"]:
            rows.append({
                "category": "global",
                "object": None,
                "query": q,
                "query_type": "trend"
            })

    df = pd.DataFrame(rows).drop_duplicates(subset=["query"]).reset_index(drop=True)
    return df

In [62]:
df_test = generate_queries_for_categories(
    config=config,
    categories=["camera_drone", "gaming"],
    include_global_terms=False
)

print("query 数量：", len(df_test))
df_test.head(20)

query 数量： 143


,category,object,query,query_type
0,generic,None,what's in my bag tech,scenario
1,generic,None,everyday carry tech EDC,scenario
2,generic,None,EDC setup gadgets,scenario
3,generic,None,tech travel setup,scenario
4,generic,None,travel gadgets packing,scenario
5,generic,None,camera bag setup,scenario
6,generic,None,drone travel setup,scenario
7,generic,None,photography gear setup,scenario
8,generic,None,minimalist tech setup,scenario
9,generic,None,desk setup tech gadgets,scenario


In [63]:
def stratified_sample_queries(df: pd.DataFrame, sample_plan: dict, random_seed: int = 42) -> pd.DataFrame:
    sampled_parts = []

    for qtype, n in sample_plan.items():
        sub = df[df["query_type"] == qtype]
        if len(sub) == 0:
            continue
        sampled_parts.append(sub.sample(n=min(n, len(sub)), random_state=random_seed))

    if not sampled_parts:
        return pd.DataFrame(columns=df.columns)

    out = pd.concat(sampled_parts, ignore_index=True).drop_duplicates(subset=["query"]).reset_index(drop=True)
    return out

In [64]:
MEMBER_ASSIGNMENT = {
    "member_1": ["consumer_electronics"],
    "member_2": ["camera_drone"],
    "member_3": ["digital_accessories"],
    "member_4": ["gaming", "valuable_items"],
    "member_5": ["travel_outdoor"]
}

In [65]:
MEMBER_SAMPLE_PLAN = {
    "object_problem": 60,
    "object_scenario": 40
}

LEADER_SAMPLE_PLAN = {
    "scenario": 50,
    "problem": 50,
    "product_support": 10,
    "trend": 10
}

In [66]:
output_dir = Path("../data/shared_data/query_assignments")
output_dir.mkdir(parents=True, exist_ok=True)

all_assignment_tables = []

for member_name, categories in MEMBER_ASSIGNMENT.items():
    df_member_full = generate_queries_for_categories(
        config=config,
        categories=categories,
        include_global_terms=False
    )

    df_member_sampled = stratified_sample_queries(
        df=df_member_full,
        sample_plan=MEMBER_SAMPLE_PLAN,
        random_seed=42
    ).copy()

    df_member_sampled["assigned_to"] = member_name

    file_path = output_dir / f"{member_name}_queries.csv"
    df_member_sampled.to_csv(file_path, index=False, encoding="utf-8-sig")

    all_assignment_tables.append(df_member_sampled)

    print(f"已生成：{file_path} | {len(df_member_sampled)} 条")

已生成：../data/shared_data/query_assignments/member_1_queries.csv | 78 条
已生成：../data/shared_data/query_assignments/member_2_queries.csv | 62 条
已生成：../data/shared_data/query_assignments/member_3_queries.csv | 88 条
已生成：../data/shared_data/query_assignments/member_4_queries.csv | 100 条
已生成：../data/shared_data/query_assignments/member_5_queries.csv | 65 条


In [67]:
df_global_full = generate_queries_for_categories(
    config=config,
    categories=[],
    include_global_terms=True
)

df_global = stratified_sample_queries(
    df=df_global_full,
    sample_plan=LEADER_SAMPLE_PLAN,
    random_seed=42
).copy()

df_global["assigned_to"] = "leader"

global_file = output_dir / f"leader_global_queries.csv"
df_global.to_csv(global_file, index=False, encoding="utf-8-sig")

print(f"已生成：{global_file} | {len(df_global)} 条")
df_global.head()

已生成：../data/shared_data/query_assignments/leader_global_queries.csv | 34 条


,category,object,query,query_type,assigned_to
0,generic,None,desk setup tech gadgets,scenario,leader
1,generic,None,gaming setup accessories,scenario,leader
2,generic,None,what's in my bag tech,scenario,leader
3,generic,None,college tech essentials,scenario,leader
4,generic,None,camera bag setup,scenario,leader


In [68]:
df_all_members = pd.concat(all_assignment_tables, ignore_index=True)
df_master = pd.concat([df_all_members, df_global], ignore_index=True)

master_file = output_dir / f"ALL_QUERY_ASSIGNMENTS.csv"
df_master.to_csv(master_file, index=False, encoding="utf-8-sig")

print("总表已导出：", master_file)
print("总 query 数：", len(df_master))
df_master.head(20)

总表已导出： ../data/shared_data/query_assignments/ALL_QUERY_ASSIGNMENTS.csv
总 query 数： 427


,category,object,query,query_type,assigned_to
0,consumer_electronics,headphones,how to store headphones,object_problem,member_1
1,consumer_electronics,macbook,macbook scratched easily,object_problem,member_1
2,consumer_electronics,macbook,safe storage for macbook,object_problem,member_1
3,consumer_electronics,earbuds,safe storage for earbuds,object_problem,member_1
4,consumer_electronics,tablet,tablet damaged in bag,object_problem,member_1
5,consumer_electronics,airpods,airpods easy to break,object_problem,member_1
6,consumer_electronics,earbuds,earbuds damaged in bag,object_problem,member_1
7,consumer_electronics,macbook,no good case for macbook,object_problem,member_1
8,consumer_electronics,airpods,safe storage for airpods,object_problem,member_1
9,consumer_electronics,ipad,ipad easy to break,object_problem,member_1


In [69]:
dup_queries = (
    df_all_members.groupby("query")
    .agg(
        assigned_count=("assigned_to", "nunique"),
        assigned_members=("assigned_to", lambda x: ", ".join(sorted(set(x))))
    )
    .reset_index()
)

dup_queries = dup_queries[dup_queries["assigned_count"] > 1].sort_values("assigned_count", ascending=False)

print("跨组员重复 query 数量：", len(dup_queries))
dup_queries.head(20)

跨组员重复 query 数量： 0


,query,assigned_count,assigned_members


In [70]:
simple_dir = output_dir / "simple_query_only"
simple_dir.mkdir(parents=True, exist_ok=True)

for member_name, categories in MEMBER_ASSIGNMENT.items():
    src_file = output_dir / f"{member_name}_queries.csv"
    df = pd.read_csv(src_file)

    simple_df = df[["query"]].drop_duplicates().reset_index(drop=True)

    dst_file = simple_dir / f"{member_name}_query_only.csv"
    simple_df.to_csv(dst_file, index=False, encoding="utf-8-sig")

    print(f"已生成简化版：{dst_file} | {len(simple_df)} 条")

leader_simple = df_global[["query"]].drop_duplicates().reset_index(drop=True)
leader_simple_file = simple_dir / f"leader_global_query_only.csv"
leader_simple.to_csv(leader_simple_file, index=False, encoding="utf-8-sig")

print(f"已生成组长简化版：{leader_simple_file} | {len(leader_simple)} 条")

已生成简化版：../data/shared_data/query_assignments/simple_query_only/member_1_query_only.csv | 78 条
已生成简化版：../data/shared_data/query_assignments/simple_query_only/member_2_query_only.csv | 62 条
已生成简化版：../data/shared_data/query_assignments/simple_query_only/member_3_query_only.csv | 88 条
已生成简化版：../data/shared_data/query_assignments/simple_query_only/member_4_query_only.csv | 100 条
已生成简化版：../data/shared_data/query_assignments/simple_query_only/member_5_query_only.csv | 65 条
已生成组长简化版：../data/shared_data/query_assignments/simple_query_only/leader_global_query_only.csv | 34 条
